# Synthetic retrospective consistency review

This notebook evaluates the transparent Protect/Grow heuristic against the supplied quarter-end outcome. The outcome is loaded separately and is never present in the runtime feature frame, score, UI, or LLM context.

These results describe one synthetic portfolio. They are not predictive validation and should not be interpreted as production model performance.

In [1]:
import numpy as np
import pandas as pd

from src.data import load_data, process_data
from src.scoring import OPERATING_HORIZON_DAYS, PROTECTION_WEIGHTS, score_accounts

raw = load_data()
scored = score_accounts(process_data())
assert "revenue_end_of_quarter" not in scored.columns
assert np.isclose(sum(PROTECTION_WEIGHTS.values()), 1.0)
assert np.allclose(
    scored["protect_value"],
    scored["current_revenue"]
    * scored["score_protection_strength"]
    * scored["score_renewal_urgency"],
)
assert np.allclose(
    scored["growth_value"],
    scored["peer_expansion_seats"]
    * scored["revenue_per_licensed_seat"]
    * scored["score_adoption_readiness"]
    * scored["score_renewal_urgency"],
)

eligible = scored["days_to_next_renewal"].le(OPERATING_HORIZON_DAYS)
evaluation = scored.loc[eligible].merge(
    raw[["account_id", "revenue_end_of_quarter"]],
    on="account_id",
    validate="one_to_one",
)
assert len(evaluation) == int(eligible.sum())
assert evaluation["days_to_next_renewal"].le(OPERATING_HORIZON_DAYS).all()
evaluation["delta"] = (
    evaluation["revenue_end_of_quarter"] - evaluation["current_revenue"]
)
evaluation["loss_dollars"] = (-evaluation["delta"]).clip(lower=0)
evaluation["growth_dollars"] = evaluation["delta"].clip(lower=0)
evaluation["change_dollars"] = evaluation["delta"].abs()

print(f"Accounts within {OPERATING_HORIZON_DAYS} days: {len(evaluation):,}")
print(evaluation["priority_action"].value_counts().to_string())

Accounts within 90 days: 244
priority_action
Protect    229
Grow        15


In [2]:
def capture_interval(selected, dollars, samples=1000, seed=42):
    selected = np.asarray(selected, dtype=bool)
    dollars = np.asarray(dollars, dtype=float)
    rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(samples):
        sample = rng.integers(0, len(dollars), len(dollars))
        denominator = dollars[sample].sum()
        estimate = dollars[sample][selected[sample]].sum() / denominator if denominator else 0
        estimates.append(estimate)
    return np.quantile(estimates, [0.025, 0.975])

rows = []
for action, value_column, outcome_column in [
    ("Protect", "protect_value", "loss_dollars"),
    ("Grow", "growth_value", "growth_dollars"),
]:
    action_accounts = evaluation[evaluation["priority_action"].eq(action)]
    selected_index = action_accounts.nlargest(50, value_column).index
    selected = evaluation.index.isin(selected_index)
    actual = evaluation[outcome_column].gt(0)
    lower, upper = capture_interval(selected, evaluation[outcome_column])
    rows.append(
        {
            "action": action,
            "assigned_accounts": len(action_accounts),
            "evaluated_top_n": int(selected.sum()),
            "outcome_precision": actual[selected].mean(),
            "outcome_recall": actual[selected].sum() / actual.sum(),
            "outcome_dollar_capture": (
                evaluation.loc[selected, outcome_column].sum()
                / evaluation[outcome_column].sum()
            ),
            "capture_95pct_low": lower,
            "capture_95pct_high": upper,
        }
    )

action_metrics = pd.DataFrame(rows).set_index("action")
action_metrics.style.format("{:.1%}", subset=[
    "outcome_precision",
    "outcome_recall",
    "outcome_dollar_capture",
    "capture_95pct_low",
    "capture_95pct_high",
])

,assigned_accounts,evaluated_top_n,outcome_precision,outcome_recall,outcome_dollar_capture,capture_95pct_low,capture_95pct_high
action,,,,,,,
Protect,229,50,16.0%,19.5%,86.5%,60.9%,93.9%
Grow,15,15,66.7%,16.7%,30.3%,6.0%,55.9%


In [3]:
ks = [10, 25, 50, 100, 200]
orders = {
    "priority_value": evaluation.sort_values("priority_value", ascending=False).index,
    "current_revenue": evaluation.sort_values("current_revenue", ascending=False).index,
    "renewal_date": evaluation.sort_values("days_to_next_renewal").index,
}

rng = np.random.default_rng(42)
rows = []
for k in ks:
    row = {"top_k": k}
    for name, order in orders.items():
        selected = evaluation.index.isin(order[:k])
        point = (
            evaluation.loc[selected, "change_dollars"].sum()
            / evaluation["change_dollars"].sum()
        )
        lower, upper = capture_interval(
            selected, evaluation["change_dollars"], seed=42 + k
        )
        row[name] = point
        row[f"{name}_95pct"] = f"{lower:.1%}–{upper:.1%}"

    random_capture = []
    for _ in range(1000):
        selected_index = rng.permutation(evaluation.index)[:k]
        random_capture.append(
            evaluation.loc[selected_index, "change_dollars"].sum()
            / evaluation["change_dollars"].sum()
        )
    row["random_mean"] = np.mean(random_capture)
    row["random_95pct"] = (
        f"{np.quantile(random_capture, 0.025):.1%}–"
        f"{np.quantile(random_capture, 0.975):.1%}"
    )
    rows.append(row)

capture_metrics = pd.DataFrame(rows).set_index("top_k")
capture_metrics.style.format("{:.1%}", subset=[
    "priority_value",
    "current_revenue",
    "renewal_date",
    "random_mean",
])

,priority_value,priority_value_95pct,current_revenue,current_revenue_95pct,renewal_date,renewal_date_95pct,random_mean,random_95pct
top_k,,,,,,,,
10,21.0%,1.5%–41.8%,30.2%,6.9%–53.8%,4.4%,0.1%–12.0%,4.2%,0.1%–16.5%
25,68.1%,44.2%–81.1%,69.1%,48.0%–83.0%,10.9%,3.3%–22.7%,10.8%,1.2%–26.6%
50,83.3%,69.9%–90.6%,85.1%,73.5%–91.6%,16.1%,6.5%–31.3%,20.5%,5.0%–41.5%
100,94.9%,90.6%–97.1%,95.6%,92.1%–97.5%,37.1%,17.7%–59.6%,41.0%,20.3%–64.7%
200,99.3%,98.6%–99.7%,99.2%,98.5%–99.6%,95.7%,90.1%–99.0%,82.0%,62.2%–96.0%


## Ablation

The progressive comparison adds renewal timing, protection signals, and the Grow motion in stages. The leave-one-signal-out comparison removes one protection component at a time, renormalizes the remaining protection weights, and leaves the growth calculation unchanged.

In [4]:
def capture_by_score(score):
    order = score.sort_values(ascending=False).index
    total_change = evaluation["change_dollars"].sum()
    return pd.Series(
        {
            k: evaluation.loc[order[:k], "change_dollars"].sum() / total_change
            for k in ks
        }
    )


progressive_scores = {
    "current_revenue": evaluation["current_revenue"],
    "revenue_x_urgency": (
        evaluation["current_revenue"] * evaluation["score_renewal_urgency"]
    ),
    "protection_only": evaluation["protect_value"],
    "full_priority": evaluation["priority_value"],
}
progressive_metrics = pd.DataFrame(
    {name: capture_by_score(score) for name, score in progressive_scores.items()}
)
progressive_metrics.index.name = "top_k"

part_columns = {name: f"score_{name}" for name in PROTECTION_WEIGHTS}
ablation_scores = {"full_priority": evaluation["priority_value"]}
for excluded in part_columns:
    remaining = [name for name in part_columns if name != excluded]
    remaining_weight = sum(PROTECTION_WEIGHTS[name] for name in remaining)
    protection_strength = sum(
        evaluation[part_columns[name]] * PROTECTION_WEIGHTS[name]
        for name in remaining
    ) / remaining_weight
    protect_value = (
        evaluation["current_revenue"]
        * protection_strength
        * evaluation["score_renewal_urgency"]
    )
    ablation_scores[f"without_{excluded}"] = pd.concat(
        [protect_value, evaluation["growth_value"]], axis=1
    ).max(axis=1)

leave_one_out_metrics = pd.DataFrame(
    {name: capture_by_score(score) for name, score in ablation_scores.items()}
)
leave_one_out_metrics.index.name = "top_k"

display(
    progressive_metrics.style.format("{:.1%}").set_caption(
        "Progressive comparison: absolute revenue-change capture"
    )
)
display(
    leave_one_out_metrics.style.format("{:.1%}").set_caption(
        "Leave-one-protection-signal-out: absolute revenue-change capture"
    )
)

,current_revenue,revenue_x_urgency,protection_only,full_priority
top_k,,,,
10,30.2%,37.1%,29.2%,21.0%
25,69.1%,63.8%,68.0%,68.1%
50,85.1%,85.1%,83.3%,83.3%
100,95.6%,95.7%,94.9%,94.9%
200,99.2%,99.2%,99.3%,99.3%


,full_priority,without_idle_seats,without_low_ai_adoption,without_support_pressure,without_contact_gap
top_k,,,,,
10,21.0%,21.0%,21.0%,21.0%,37.5%
25,68.1%,65.0%,61.8%,65.0%,68.1%
50,83.3%,79.6%,83.3%,83.3%,83.3%
100,94.9%,94.6%,94.9%,94.9%,95.6%
200,99.3%,99.3%,99.2%,99.3%,99.3%


## Interpretation

- All comparisons use the same 244 accounts the product can surface: renewals within the 90-day operating horizon.
- Revenue times renewal urgency captures the most absolute change at the top-10 budget, while current revenue narrowly leads the full heuristic from 25 to 100 accounts. The full heuristic is retained for its actionable Protect/Grow reasoning, not because this retrospective proves better generalization.
- The leave-one-out results vary by review budget: removing contact gap changes the top-10 ranking most, while removing AI adoption has the largest effect at top 25. No single component dominates consistently enough to justify tuning on this synthetic outcome.
- Bootstrap intervals quantify sampling uncertainty within this portfolio, but cannot correct synthetic-data bias or establish out-of-sample validity. Better performance on future data remains a hypothesis requiring historical snapshots and time-based validation.